In [0]:
import mlflow, pandas as pd, mlflow.deployments

# 1. Load the model to get the EXACT columns it expects
model = mlflow.sklearn.load_model("models:/northstar.ml.late_delivery@champion")
expected_cols = list(model.feature_names_in_)

# 2. Build a sample input, aligned to those columns
sample = spark.read.table("northstar.gold.ml_features_late_delivery").limit(3).toPandas()
sample = pd.get_dummies(sample, columns=["customer_state"], drop_first=True)
X = (sample.drop(columns=["order_id", "is_late"], errors="ignore")
           .reindex(columns=expected_cols, fill_value=0)
           .astype("float64"))

# 3. Call the serving endpoint
client = mlflow.deployments.get_deploy_client("databricks")
resp = client.predict(
    endpoint="late-delivery",
    inputs={"dataframe_split": {"columns": X.columns.tolist(), "data": X.values.tolist()}})
print("Predictions:", resp)